In [0]:
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA default")

PREFIX = "wdatt_movie_"
BASE = "/Volumes/workspace/default/movie_recommender"

display(dbutils.fs.ls(BASE))



In [0]:
bronze_tables = [
    f"{PREFIX}bronze_movies",
    f"{PREFIX}bronze_links",
    f"{PREFIX}bronze_tags",
    f"{PREFIX}bronze_scraped_metadata",
    f"{PREFIX}bronze_ratings_top500",
    f"{PREFIX}bronze_ratings_raw",
]

for t in bronze_tables:
    spark.sql(f"DROP TABLE IF EXISTS {t}")

print("✅ Dropped (if existed):", bronze_tables)


In [0]:
movies_path  = f"{BASE}/movies.csv"
links_path   = f"{BASE}/links.csv"
tags_path    = f"{BASE}/tags.csv"
ratings_path = f"{BASE}/ratings.csv"
scraped_path = f"{BASE}/scraped_metadata.json"

df_movies  = spark.read.option("header", True).option("inferSchema", True).csv(movies_path)
df_links   = spark.read.option("header", True).option("inferSchema", True).csv(links_path)
df_tags    = spark.read.option("header", True).option("inferSchema", True).csv(tags_path)
df_ratings = spark.read.option("header", True).option("inferSchema", True).csv(ratings_path)
df_scraped = spark.read.option("multiLine", True).json(scraped_path)

print("✅ Loaded raw:")
print("movies :", df_movies.count())
print("links  :", df_links.count())
print("tags   :", df_tags.count())
print("ratings:", df_ratings.count())
print("scraped:", df_scraped.count())
df_scraped.printSchema()


In [0]:
from pyspark.sql import functions as F

# Write raw (except ratings raw is optional, but we’ll write then filter)
df_movies.write.format("delta").mode("overwrite").saveAsTable(f"{PREFIX}bronze_movies")
df_links.write.format("delta").mode("overwrite").saveAsTable(f"{PREFIX}bronze_links")
df_tags.write.format("delta").mode("overwrite").saveAsTable(f"{PREFIX}bronze_tags")
df_scraped.write.format("delta").mode("overwrite").saveAsTable(f"{PREFIX}bronze_scraped_metadata")

# Filter ratings to only scraped movieIds (top500)
scraped_ids = spark.table(f"{PREFIX}bronze_scraped_metadata").select(F.col("movieId")).where("movieId is not null").distinct()

ratings_top500 = (
    df_ratings
    .select("userId","movieId","rating","timestamp")
    .join(scraped_ids, on="movieId", how="inner")
)

ratings_top500.write.format("delta").mode("overwrite").saveAsTable(f"{PREFIX}bronze_ratings_top500")

print("✅ Bronze written")
for t in ["bronze_movies","bronze_links","bronze_tags","bronze_scraped_metadata","bronze_ratings_top500"]:
    print(f"{PREFIX}{t} rows:", spark.table(f"{PREFIX}{t}").count())


In [0]:
assert spark.table(f"{PREFIX}bronze_movies").count() > 0
assert spark.table(f"{PREFIX}bronze_links").count() > 0
assert spark.table(f"{PREFIX}bronze_tags").count() > 0
assert spark.table(f"{PREFIX}bronze_scraped_metadata").count() >= 500
assert spark.table(f"{PREFIX}bronze_ratings_top500").count() > 0
print("✅ BRONZE validated")


In [0]:
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA default")
spark.sql("DROP TABLE IF EXISTS wdatt_movie_bronze_ratings_raw")
print("Dropped wdatt_movie_bronze_ratings_raw")
